In [1]:
print('hello world')

hello world


In [31]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.svm import SVC
from joblib import Parallel, delayed
import matplotlib.pyplot as plt 
import seaborn as sns
%matplotlib inline
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler
import scipy.stats as stats
from sklearn.metrics import accuracy_score
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, FunctionTransformer, LabelEncoder,PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [32]:
df_2022=pd.read_csv('fifa_ranking_2022-10-06.csv')
df_2025=pd.read_csv('fifa_ranking_2026-06-08.csv')
df_matches=pd.read_csv('matches_1930_2022.csv')
df_shedule=pd.read_csv('schedule_2026.csv')
df_world_cup=pd.read_csv('world_cup.csv')



In [33]:
#trying to find total teams from 1930 to 2022
match_teams = set(df_matches['home_team'].dropna().unique()).union(set(df_matches['away_team'].dropna().unique()))

# total teams in 2022 
ranking_teams = set(df_2022['team'].dropna().unique())

# team that are not in 2022 bit were there in btw 1930 to 2022
missing_in_rankings = sorted(list(match_teams - ranking_teams))

print(f"Total unique teams in Matches: {len(match_teams)}")
print(f"Total unique teams in Rankings: {len(ranking_teams)}")
print(f"Number of mismatches: {len(missing_in_rankings)}")
print("Mismatched Teams:", missing_in_rankings)

Total unique teams in Matches: 86
Total unique teams in Rankings: 211
Number of mismatches: 11
Mismatched Teams: ['Czech Republic', 'Czechoslovakia', 'Dutch East Indies', 'FR Yugoslavia', 'Germany DR', 'Serbia and Montenegro', 'Soviet Union', 'United States', 'West Germany', 'Yugoslavia', 'Zaire']


In [34]:
missing_in_rankings



['Czech Republic',
 'Czechoslovakia',
 'Dutch East Indies',
 'FR Yugoslavia',
 'Germany DR',
 'Serbia and Montenegro',
 'Soviet Union',
 'United States',
 'West Germany',
 'Yugoslavia',
 'Zaire']

In [35]:
country_mapping = {
    'United States': 'USA',
    'Czech Republic': 'Czechia',
    'Czechoslovakia': 'Czechia',
    'Dutch East Indies': 'Indonesia',
    'FR Yugoslavia': 'Serbia',
    'Germany DR': 'Germany',
    'Serbia and Montenegro': 'Serbia',
    'Soviet Union': 'Russia',
    'West Germany': 'Germany',
    'Yugoslavia': 'Serbia',
    'Zaire': 'Congo DR'
}

df_matches['home_team'] = df_matches['home_team'].replace(country_mapping)
df_matches['away_team'] = df_matches['away_team'].replace(country_mapping)

In [36]:
print(df_matches['home_penalty'].value_counts(dropna=False))
print(df_matches['away_penalty'].value_counts(dropna=False))

home_penalty
NaN    929
3.0     13
4.0     11
5.0      5
1.0      3
2.0      2
0.0      1
Name: count, dtype: int64
away_penalty
NaN    929
3.0     11
4.0     10
2.0      9
5.0      3
0.0      1
1.0      1
Name: count, dtype: int64


 ## Rounds where a draw is allowed
draw_rounds = [
   'Group stage',
    'First group stage',
    'Second group stage']
#
def get_match_result(row):
    # Normal time / Extra time result
    if row['home_score'] > row['away_score']:
          return 'Home Win'
    elif row['home_score'] < row['away_score']:
       return 'Away Win'
    # Scores are equal
    else:
        # Draw is allowed in these rounds
        if row['Round'] in draw_rounds:
            return 'Draw'
        # Knockout rounds -> decide by penalties
        else:
            if pd.notna(row['home_penalty']) and pd.notna(row['away_penalty']):
                if row['home_penalty'] > row['away_penalty']:
                    return 'Home Win'
                elif row['home_penalty'] < row['away_penalty']:
                    return 'Away Win'
            #If penalty data is unavailable
            return 'Draw'


 #Create the new column
df_matches['Match_Result'] = df_matches.apply(get_match_result, axis=1)

 #Check the result distribution
print(df_matches['Match_Result'].value_counts())

"While technically precise to implement the above code—given that match outcomes vary between round-robin stages (which allow draws) and knockout phases (which do not)—the simpler logic below will be utilized for this model.

In [37]:
df_matches['Match_Result'] = np.where(
    df_matches['home_score'] > df_matches['away_score'], 0,
    np.where(
        df_matches['home_score'] < df_matches['away_score'], 1,
        2
    )
)
print(df_matches['Match_Result'].value_counts())

Match_Result
0    532
1    218
2    214
Name: count, dtype: int64


In [38]:

train_data = df_matches[df_matches['Year'] <= 2018].copy()
test_data = df_matches[df_matches['Year'] == 2022].copy()

Since we cannot use home_score and away_score in the model as practically we dont know the scores before the match it will be a data leakage so we will use a new column that is the rolling average of goals scored up to that point in time

In [39]:
train_data['Date'] = pd.to_datetime(train_data['Date'])
test_data['Date'] = pd.to_datetime(test_data['Date'])

In [40]:
home_goals = train_data[['home_team', 'home_score']].rename(
    columns={'home_team': 'team', 'home_score': 'goals_scored'}
)

away_goals = train_data[['away_team', 'away_score']].rename(
    columns={'away_team': 'team', 'away_score': 'goals_scored'}
)

all_goals = pd.concat([home_goals, away_goals], ignore_index=True)

team_avg_goals = all_goals.groupby('team')['goals_scored'].mean()

train_data['home_avg_goals'] = train_data['home_team'].map(team_avg_goals)
train_data['away_avg_goals'] = train_data['away_team'].map(team_avg_goals)
test_data['home_avg_goals'] = test_data['home_team'].map(team_avg_goals)
test_data['away_avg_goals'] = test_data['away_team'].map(team_avg_goals)

overall_avg = all_goals['goals_scored'].mean()

train_data[['home_avg_goals', 'away_avg_goals']] = (
    train_data[['home_avg_goals', 'away_avg_goals']].fillna(overall_avg)
)

test_data[['home_avg_goals', 'away_avg_goals']] = (
    test_data[['home_avg_goals', 'away_avg_goals']].fillna(overall_avg)
)


Iintroducing a new feature to capture each team's historical World Cup experience. In international football, experienced teams typically possess a higher probability of winning because they have adapted to tournament pressure and learned from past mistakes—an advantage that debutant or less-experienced teams do not share.

In [41]:
import pandas as pd

# 1. Isolate the Years and Teams from the historical training data (1930 - 2018)
home_years = train_data[['Year', 'home_team']].rename(columns={'home_team': 'team'})
away_years = train_data[['Year', 'away_team']].rename(columns={'away_team': 'team'})

# 2. Combine them and drop duplicates
# A team plays multiple matches in a single World Cup, so we drop duplicates 
# to ensure we only count '1' appearance per tournament year.
all_appearances = pd.concat([home_years, away_years]).drop_duplicates()

# 3. Count the unique years for each team to get their Total Experience
world_cup_experience = all_appearances.groupby('team')['Year'].count()

# 4. Map the experience back to the Training Data
train_data['home_wc_experience'] = train_data['home_team'].map(world_cup_experience)
train_data['away_wc_experience'] = train_data['away_team'].map(world_cup_experience)

# 5. Map the experience back to the Testing Data (2022)
test_data['home_wc_experience'] = test_data['home_team'].map(world_cup_experience)
test_data['away_wc_experience'] = test_data['away_team'].map(world_cup_experience)

# 6. Handle the Debuting Teams (Like Qatar!)
# If a team isn't in the 1930-2018 data, they have 0 prior experience.
# Unlike the 'average goals' feature, filling with 0 is mathematically perfect here!
train_data[['home_wc_experience', 'away_wc_experience']] = (
    train_data[['home_wc_experience', 'away_wc_experience']].fillna(0)
)
test_data[['home_wc_experience', 'away_wc_experience']] = (
    test_data[['home_wc_experience', 'away_wc_experience']].fillna(0)
)

"We are factoring in a Home Field Advantage weight into our model. Teams playing in their host country benefit from both familiarity with local pitches and significant crowd support. While stadium attendance consists of opposing fans as well, we assume that total attendance scales proportionally with home-team support, amplifying their advantage as match attendance increases."

In [42]:

train_data['home_is_host'] = (train_data['home_team'] == train_data['Host']).astype(int)
train_data['away_is_host'] = (train_data['away_team'] == train_data['Host']).astype(int)

test_data['home_is_host'] = (test_data['home_team'] == test_data['Host']).astype(int)
test_data['away_is_host'] = (test_data['away_team'] == test_data['Host']).astype(int)

train_data['home_adrenaline_boost'] = train_data['home_is_host'] 
train_data['away_adrenaline_boost'] = train_data['away_is_host'] 

test_data['home_adrenaline_boost'] = test_data['home_is_host'] 
test_data['away_adrenaline_boost'] = test_data['away_is_host'] 

train_data[['home_adrenaline_boost', 'away_adrenaline_boost']] = (
    train_data[['home_adrenaline_boost', 'away_adrenaline_boost']].fillna(0)
)

test_data[['home_adrenaline_boost', 'away_adrenaline_boost']] = (
    test_data[['home_adrenaline_boost', 'away_adrenaline_boost']].fillna(0)
)


"To avoid data leakage, we cannot directly include match-specific penalty outcomes in our feature set. Instead, we capture a team's proficiency by engineering a feature for total historical penalties played. Teams with higher penalty experience are statistically better prepared to handle high-pressure shootout scenarios, giving them a distinct advantage if a match transitions to penalties."

In [43]:
import pandas as pd

# Count a shootout if the penalty value is not null
home_penalty_exp = (
    train_data[train_data['home_penalty'].notna()]
    .groupby('home_team')
    .size()
)

away_penalty_exp = (
    train_data[train_data['away_penalty'].notna()]
    .groupby('away_team')
    .size()
)

# Total shootout experience (home + away)
team_penalty_exp = home_penalty_exp.add(
    away_penalty_exp,
    fill_value=0
)

# Map to training data
train_data['home_penalty_exp'] = (
    train_data['home_team']
    .map(team_penalty_exp)
    .fillna(0)
)

train_data['away_penalty_exp'] = (
    train_data['away_team']
    .map(team_penalty_exp)
    .fillna(0)
)

# Map to test data
test_data['home_penalty_exp'] = (
    test_data['home_team']
    .map(team_penalty_exp)
    .fillna(0)
)

test_data['away_penalty_exp'] = (
    test_data['away_team']
    .map(team_penalty_exp)
    .fillna(0)
)

In [44]:
train_data['home_yellow_red_card'].dropna().sample(5).tolist()

['John Heitinga · 109',
 'Beto · 66',
 'Cyril Domoraud · 90+2',
 'Dario Šimić · 85|Josip Šimunić · 90+3',
 'Avery John · 46']

In [45]:
train_data['home_red_card'].sample(10)

527                     NaN
214                     NaN
143                     NaN
819    Vladica Popović · 71
266    Marco Materazzi · 50
834                     NaN
721                     NaN
823                     NaN
364                     NaN
454                     NaN
Name: home_red_card, dtype: str

"Since in-match events like yellow, red, and yellow-red cards cannot be used directly in the model, 
I engineered a consolidated 'discipline score.' This new feature aggregates the total
number of cards to effectively quantify player misconduct and team aggression

In [46]:
import pandas as pd

# 1. Optimized Vectorized Card Counter
def count_cards_vectorized(series, delimiter=';'):
    # Fill NA with empty string, convert to string, split, and count non-empty elements
    return series.fillna('').astype(str).apply(lambda x: len([i for i in x.split(delimiter) if i.strip()]) if x else 0)

# If your red cards are actually separated by semicolons, change '.' to ';' below
card_cols = {
    'home_yellow_cards': ('home_yellow_card_long', ';'),
    'away_yellow_cards': ('away_yellow_card_long', ';'),
    'home_red_cards': ('home_red_card', '.'),          # Double check if '.' is correct!
    'away_red_cards': ('away_red_card', '.'),
    'home_yellow_red_cards': ('home_yellow_red_card', '.'),
    'away_yellow_red_cards': ('away_yellow_red_card', '.')
}

for new_col, (raw_col, sep) in card_cols.items():
    train_data[new_col] = count_cards_vectorized(train_data[raw_col], sep)

# 2. Calculate Match Card Scores
for side in ['home', 'away']:
    train_data[f'{side}_match_card_score'] = (
        train_data[f'{side}_yellow_cards']
        + 2 * train_data[f'{side}_red_cards']
        + 2 * train_data[f'{side}_yellow_red_cards']
    )

# 3. Create Global Team Discipline Reference
home_cards = train_data[['home_team', 'home_match_card_score']].rename(columns={'home_team': 'team', 'home_match_card_score': 'card_score'})
away_cards = train_data[['away_team', 'away_match_card_score']].rename(columns={'away_team': 'team', 'away_match_card_score': 'card_score'})

team_discipline_score = pd.concat([home_cards, away_cards], ignore_index=True).groupby('team')['card_score'].mean()
global_mean = team_discipline_score.mean()

# 4. Map Scores to Train and Test Sets
for side in ['home', 'away']:
    train_data[f'{side}_discipline_score'] = train_data[f'{side}_team'].map(team_discipline_score)
    test_data[f'{side}_discipline_score'] = test_data[f'{side}_team'].map(team_discipline_score).fillna(global_mean)

in the below code we are giving each team a unique id (number) as when we do label encoding or any encoding in away and home team the same team assume brazil is getting different number which is not pretty good exercise .

In [47]:
# Get all unique teams from both columns
all_teams = pd.concat([
    train_data['home_team'],
    train_data['away_team']
]).unique()

# Create mapping
team_to_id = {team: idx for idx, team in enumerate(sorted(all_teams))}

# Apply to train
train_data['home_team_id'] = train_data['home_team'].map(team_to_id)
train_data['away_team_id'] = train_data['away_team'].map(team_to_id)

# Apply to test
test_data['home_team_id'] = test_data['home_team'].map(team_to_id).fillna(-1).astype(int)
test_data['away_team_id'] = test_data['away_team'].map(team_to_id).fillna(-1).astype(int)

In [ ]:
train_data['Venue_City'] = train_data['Venue'].str.split(',').str[-1].str.strip()
test_data['Venue_City'] = test_data['Venue'].str.split(',').str[-1].str.strip()

train_data['Referee'] = train_data['Referee'].fillna('Unknown')

test_data['Referee'] = test_data['Referee'].fillna('Unknown')

In [49]:
test_data.columns

Index(['home_team', 'away_team', 'home_score', 'home_xg', 'home_penalty',
       'away_score', 'away_xg', 'away_penalty', 'home_manager', 'home_captain',
       'away_manager', 'away_captain', 'Attendance', 'Venue', 'Officials',
       'Round', 'Date', 'Score', 'Referee', 'Notes', 'Host', 'Year',
       'home_goal', 'away_goal', 'home_goal_long', 'away_goal_long',
       'home_own_goal', 'away_own_goal', 'home_penalty_goal',
       'away_penalty_goal', 'home_penalty_miss_long', 'away_penalty_miss_long',
       'home_penalty_shootout_goal_long', 'away_penalty_shootout_goal_long',
       'home_penalty_shootout_miss_long', 'away_penalty_shootout_miss_long',
       'home_red_card', 'away_red_card', 'home_yellow_red_card',
       'away_yellow_red_card', 'home_yellow_card_long',
       'away_yellow_card_long', 'home_substitute_in_long',
       'away_substitute_in_long', 'Match_Result', 'home_avg_goals',
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host

In [50]:
used_cols=['home_xg','away_xg','home_team_id','home_manager', 'away_manager','away_team_id',
           'Match_Result', 'home_avg_goals','Attendance', 'Venue_City','Round', 
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host', 'away_is_host', 'home_adrenaline_boost','Referee',
       'away_adrenaline_boost', 'home_penalty_exp', 'away_penalty_exp', 'Host', 'Year',
       'home_discipline_score', 'away_discipline_score']

In [51]:

X_train = train_data[used_cols]
y_train = train_data['Match_Result']

# Create Testing Inputs and Targets (2022 World Cup)
X_test = test_data[used_cols]
y_test = test_data['Match_Result']



In [23]:
label_cols=['away_manager','Venue_City','home_manager','Referee']


to_stande=['home_xg','away_xg', 'home_avg_goals','Attendance', 
       'away_avg_goals', 'home_wc_experience', 'away_wc_experience',
       'home_is_host', 'away_is_host', 'home_adrenaline_boost','home_team_id','away_team_id',
       'away_adrenaline_boost', 'home_penalty_exp', 'away_penalty_exp', 'Year',
       'home_discipline_score', 'away_discipline_score']

onehot_cols = [
    'Round',
    'Host'
]


# ----------------------------
# Column Transformer
# ----------------------------

preprocessor = ColumnTransformer(
    transformers=[

        (
            'ordinal',
            OrdinalEncoder(
                handle_unknown='use_encoded_value',
                unknown_value=-1
            ),
            label_cols
        ),

        (
            'onehot',
            OneHotEncoder(
                handle_unknown='ignore'
            ),
            onehot_cols
        ),

        (
            'numeric',
            StandardScaler(),
            to_stande
        )

    ],
    remainder='drop'
)

# ----------------------------
# Complete Pipeline
# -------------------------
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [54]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV

# Decision Tree
dt = DecisionTreeClassifier(random_state=42)

# Parameter Grid
param_grid = {
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 5, 10, 15, 20, 30],
    'min_samples_split': [2, 5, 10, 20],
    'min_samples_leaf': [1, 2, 4, 8],
    'max_features': [None, 'sqrt', 'log2']
}

# Grid Search
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',      # or 'f1_weighted'
    n_jobs=-1,
    verbose=2
)

# Train
grid_search.fit(X_train_processed, y_train)

# Best Model
best_dt = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Accuracy:")
print(grid_search.best_score_)

# Prediction
y_pred = best_dt.predict(X_test_processed)

Fitting 5 folds for each of 864 candidates, totalling 4320 fits
Best Parameters:
{'criterion': 'gini', 'max_depth': 5, 'max_features': 'log2', 'min_samples_leaf': 1, 'min_samples_split': 20}

Best CV Accuracy:
0.5211111111111111


In [56]:
print(classification_report(y_test,y_pred))

              precision    recall  f1-score   support

           0       0.53      0.69      0.60        29
           1       0.42      0.55      0.48        20
           2       0.00      0.00      0.00        15

    accuracy                           0.48        64
   macro avg       0.32      0.41      0.36        64
weighted avg       0.37      0.48      0.42        64



c:\Users\Administrator\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [29]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Random Forest
rf = RandomForestClassifier(random_state=42)

# Parameter Grid
param_grid = {
    'n_estimators': [100, 200, 300],
    'criterion': ['gini', 'entropy', 'log_loss'],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False],
    'class_weight': ['balanced']
}

# Grid Search
grid_search = GridSearchCV(
    estimator=rf,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)

# Train
grid_search.fit(X_train_processed, y_train)

# Best Model
best_rf = grid_search.best_estimator_

print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest CV Accuracy:")
print(grid_search.best_score_)

# Prediction
y_pred = best_rf.predict(X_test_processed)

Fitting 5 folds for each of 1296 candidates, totalling 6480 fits
Best Parameters:
{'bootstrap': False, 'class_weight': 'balanced', 'criterion': 'gini', 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}

Best CV Accuracy:
0.3377777777777778


In [90]:
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.65      0.75      0.70        32
           1       0.52      0.64      0.57        22
           2       0.00      0.00      0.00        10

    accuracy                           0.59        64
   macro avg       0.39      0.46      0.42        64
weighted avg       0.50      0.59      0.54        64



c:\Users\Administrator\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Administrator\anaconda3\envs\myenv\Lib\site-packages\sklearn\metrics\_classification.py:1879: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capital

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report



# Calculate individual metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# Print out the results cleanly
print("🏆 --- Model Evaluation on 2022 World Cup --- 🏆")
print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1-Score:  {f1:.4f}\n")

# Generate the full classification report for a deep dive into each class
print("📊 --- Detailed Classification Report --- 📊")
print(classification_report(y_test, y_pred, target_names=['Home Win (0)', 'Away Win (1)', 'Draw (2)'], zero_division=0))

🏆 --- Model Evaluation on 2022 World Cup --- 🏆
Accuracy:        0.5469 (54.69%)
Macro Precision: 0.4748
Macro Recall:    0.4751
Macro F1-Score:  0.4405

📊 --- Detailed Classification Report --- 📊
              precision    recall  f1-score   support

Home Win (0)       0.61      0.76      0.68        29
Away Win (1)       0.48      0.60      0.53        20
    Draw (2)       0.33      0.07      0.11        15

    accuracy                           0.55        64
   macro avg       0.47      0.48      0.44        64
weighted avg       0.51      0.55      0.50        64



In [58]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create the model
xgb = XGBClassifier(
    objective='multi:softprob',   # Multi-class classification
    num_class=3,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0,
    reg_lambda=1,
    random_state=42,
    eval_metric='mlogloss'
)

# Train
best_xgb=xgb.fit(X_train_processed, y_train)

# Predict
y_pred = xgb.predict(X_test_processed)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.625
              precision    recall  f1-score   support

           0       0.69      0.83      0.75        29
           1       0.52      0.70      0.60        20
           2       1.00      0.13      0.24        15

    accuracy                           0.62        64
   macro avg       0.73      0.55      0.53        64
weighted avg       0.71      0.62      0.58        64



i got the above hyper parameters that were giving me the best results . without hyperparameters
i was getting an accuracy of around .52 and now i am getting .62

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.utils.class_weight import compute_sample_weight

# -----------------------------------
# Compute Sample Weights
# -----------------------------------
sample_weights = compute_sample_weight(
    class_weight='balanced',
    y=y_train
)

xgb = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    random_state=42,
    eval_metric='mlogloss'
)

param_grid = {
    'n_estimators': [100, 150, 200, 300],
    'max_depth': [3, 5, 7],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.8, 1.0],
    'colsample_bytree': [0.8, 1.0],
    'min_child_weight': [1, 3, 5]
}


grid = GridSearchCV(
    estimator=xgb,
    param_grid=param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1,
    verbose=2
)


grid.fit(
    X_train_processed,
    y_train,
    sample_weight=sample_weights
)

# -----------------------------------
# Best Model
# -----------------------------------
best_xgb = grid.best_estimator_

print("Best Parameters:")
print(grid.best_params_)

print("\nBest CV Accuracy:")
print(grid.best_score_)

# -----------------------------------
# Prediction
# -----------------------------------
y_pred = best_xgb.predict(X_test_processed)

Fitting 5 folds for each of 432 candidates, totalling 2160 fits
Best Parameters:
{'colsample_bytree': 0.8, 'learning_rate': 0.01, 'max_depth': 3, 'min_child_weight': 5, 'n_estimators': 100, 'subsample': 0.8}

Best CV Accuracy:
0.27528851912987556


In [ ]:
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

# Create the model
xgb = XGBClassifier(
    objective='multi:softprob',   # Multi-class classification
    num_class=3,
    n_estimators=300,
    learning_rate=0.05,
    max_depth=5,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    gamma=0.1,
    reg_alpha=0,
    reg_lambda=1,
    random_state=42,
    eval_metric='mlogloss'
)

# Train
best_xgb=xgb.fit(X_train_processed, y_train)

# Predict
y_pred = xgb.predict(X_test_processed)

# Evaluate
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

In [56]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Assuming you already made your predictions:
# y_pred = best_xgb_model.predict(X_test)

# Calculate individual metrics
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred, average='macro', zero_division=0)
recall = recall_score(y_test, y_pred, average='macro', zero_division=0)
f1 = f1_score(y_test, y_pred, average='macro', zero_division=0)

# Print out the results cleanly
print("🏆 --- Model Evaluation on 2022 World Cup --- 🏆")
print(f"Accuracy:        {accuracy:.4f} ({accuracy*100:.2f}%)")
print(f"Macro Precision: {precision:.4f}")
print(f"Macro Recall:    {recall:.4f}")
print(f"Macro F1-Score:  {f1:.4f}\n")

# Generate the full classification report for a deep dive into each class
print("📊 --- Detailed Classification Report --- 📊")
print(classification_report(y_test, y_pred, target_names=['Home Win (0)', 'Away Win (1)', 'Draw (2)'], zero_division=0))

🏆 --- Model Evaluation on 2022 World Cup --- 🏆
Accuracy:        0.6250 (62.50%)
Macro Precision: 0.7347
Macro Recall:    0.5536
Macro F1-Score:  0.5270

📊 --- Detailed Classification Report --- 📊
              precision    recall  f1-score   support

Home Win (0)       0.69      0.83      0.75        29
Away Win (1)       0.52      0.70      0.60        20
    Draw (2)       1.00      0.13      0.24        15

    accuracy                           0.62        64
   macro avg       0.73      0.55      0.53        64
weighted avg       0.71      0.62      0.58        64



In [59]:
import pandas as pd
import numpy as np

# =========================================================
# 1. BUILD THE LOOKUP DICTIONARIES (HOME + AWAY COMBINED)
# =========================================================
# We combine home and away data to ensure we capture the most 
# accurate and recent historical statistics for every single team.

# --- Average Goals ---
home_goals = train_data[['home_team', 'home_avg_goals']].rename(columns={'home_team': 'team', 'home_avg_goals': 'stat'})
away_goals = train_data[['away_team', 'away_avg_goals']].rename(columns={'away_team': 'team', 'away_avg_goals': 'stat'})
team_avg_goals_dict = pd.concat([home_goals, away_goals]).groupby('team')['stat'].last().to_dict()
global_avg_goals = train_data['home_avg_goals'].mean()

# --- World Cup Experience ---
home_exp = train_data[['home_team', 'home_wc_experience']].rename(columns={'home_team': 'team', 'home_wc_experience': 'stat'})
away_exp = train_data[['away_team', 'away_wc_experience']].rename(columns={'away_team': 'team', 'away_wc_experience': 'stat'})
team_wc_exp_dict = pd.concat([home_exp, away_exp]).groupby('team')['stat'].max().to_dict()

# --- Penalty Experience ---
home_pen = train_data[['home_team', 'home_penalty_exp']].rename(columns={'home_team': 'team', 'home_penalty_exp': 'stat'})
away_pen = train_data[['away_team', 'away_penalty_exp']].rename(columns={'away_team': 'team', 'away_penalty_exp': 'stat'})
team_pen_exp_dict = pd.concat([home_pen, away_pen]).groupby('team')['stat'].max().to_dict()

# --- Discipline Score ---
home_disc = train_data[['home_team', 'home_discipline_score']].rename(columns={'home_team': 'team', 'home_discipline_score': 'stat'})
away_disc = train_data[['away_team', 'away_discipline_score']].rename(columns={'away_team': 'team', 'away_discipline_score': 'stat'})
team_disc_score_dict = pd.concat([home_disc, away_disc]).groupby('team')['stat'].last().to_dict()
global_disc_score = train_data['home_discipline_score'].mean()

# --- Global Defaults for match-specific unknowns ---
median_xg = train_data['home_xg'].median() if 'home_xg' in train_data.columns else 1.2
median_attendance = train_data['Attendance'].median()


# =========================================================
# 2. THE PREDICTION FUNCTION
# =========================================================
def predict_match(home_team, away_team, host_country="Unknown", year=2026, round_name="Group stage"):
    """
    Predicts the outcome of a match using only the team names.
    Automatically fetches required features and formats the pipeline input.
    """
    
    # 1. Fetch mapping IDs (defaults to -1 if team is completely new)
    h_id = team_to_id.get(home_team, -1)
    a_id = team_to_id.get(away_team, -1)
    
    # 2. Build the 1-row DataFrame with all 24 features your model expects
    match_data = pd.DataFrame({
        'home_xg': [median_xg],  
        'away_xg': [median_xg],
        'home_team_id': [h_id],
        'home_manager': ['Unknown'], # Handled safely by OrdinalEncoder(unknown_value=-1)
        'away_manager': ['Unknown'],
        'away_team_id': [a_id],
        
        # Look up historical stats, use global averages if team has no history
        'home_avg_goals': [team_avg_goals_dict.get(home_team, global_avg_goals)],
        'away_avg_goals': [team_avg_goals_dict.get(away_team, global_avg_goals)],
        'home_wc_experience': [team_wc_exp_dict.get(home_team, 0)],
        'away_wc_experience': [team_wc_exp_dict.get(away_team, 0)],
        'home_penalty_exp': [team_pen_exp_dict.get(home_team, 0)],
        'away_penalty_exp': [team_pen_exp_dict.get(away_team, 0)],
        'home_discipline_score': [team_disc_score_dict.get(home_team, global_disc_score)],
        'away_discipline_score': [team_disc_score_dict.get(away_team, global_disc_score)],
        
        # Match Specifics
        'Attendance': [median_attendance],
        'Venue_City': ['Unknown'], 
        'Round': [round_name],
        'Host': [host_country],
        'Year': [year],
        'Referee': ['Unknown'],
        
        # Home field advantage logic built into your notebook
        'home_is_host': [1 if home_team == host_country else 0],
        'away_is_host': [1 if away_team == host_country else 0],
        'home_adrenaline_boost': [1 if home_team == host_country else 0],
        'away_adrenaline_boost': [1 if away_team == host_country else 0]
    })
    
    # 3. Pass through the ColumnTransformer (preprocessor)
    try:
        processed_data = preprocessor.transform(match_data)
    except Exception as e:
        return f"Error in preprocessing: {e}"
    
    # 4. Get probabilities from your best model
    probs = best_xgb.predict_proba(processed_data)[0]
    
    # Target Map: 0 = Home Win, 1 = Away Win, 2 = Draw
    home_win_pct = probs[0] * 100
    away_win_pct = probs[1] * 100
    draw_pct = probs[2] * 100
    
    # 5. Print beautifully formatted results
    print(f"⚽ Match Prediction: {home_team} (Home) vs {away_team} (Away) ⚽")
    print("-" * 50)
    print(f"🏆 {home_team} Win: {home_win_pct:.2f}%")
    print(f"🤝 Draw:          {draw_pct:.2f}%")
    print(f"🏆 {away_team} Win: {away_win_pct:.2f}%")
    print("-" * 50)
    
    # Return the raw dictionary for further programmatic use
    return {
        "Home_Win": home_win_pct, 
        "Away_Win": away_win_pct, 
        "Draw": draw_pct
    }

# ==========================================
# TEST THE FUNCTION
# ==========================================
# Run a quick test to make sure it works perfectly!
predict_match('Brazil', 'Argentina')

⚽ Match Prediction: Brazil (Home) vs Argentina (Away) ⚽
--------------------------------------------------
🏆 Brazil Win: 25.51%
🤝 Draw:          48.80%
🏆 Argentina Win: 25.69%
--------------------------------------------------


{'Home_Win': np.float32(25.51402),
 'Away_Win': np.float32(25.685087),
 'Draw': np.float32(48.80089)}

In [60]:
oho=df_shedule[['home_team','away_team']].head(10)

In [61]:
for home_team, away_team in zip(oho['home_team'], oho['away_team']):
    predict_match(home_team, away_team)

⚽ Match Prediction: Mexico (Home) vs South Africa (Away) ⚽
--------------------------------------------------
🏆 Mexico Win: 21.39%
🤝 Draw:          33.11%
🏆 South Africa Win: 45.50%
--------------------------------------------------
⚽ Match Prediction: Korea Republic (Home) vs Czechia (Away) ⚽
--------------------------------------------------
🏆 Korea Republic Win: 42.85%
🤝 Draw:          7.51%
🏆 Czechia Win: 49.64%
--------------------------------------------------
⚽ Match Prediction: Canada (Home) vs Bosnia-Herzegovina (Away) ⚽
--------------------------------------------------
🏆 Canada Win: 6.32%
🤝 Draw:          7.52%
🏆 Bosnia-Herzegovina Win: 86.16%
--------------------------------------------------
⚽ Match Prediction: United States (Home) vs Paraguay (Away) ⚽
--------------------------------------------------
🏆 United States Win: 33.20%
🤝 Draw:          7.78%
🏆 Paraguay Win: 59.02%
--------------------------------------------------
⚽ Match Prediction: Qatar (Home) vs Switzerland 

In [64]:
train_data['Host'].nunique()

16

#FROM HERE REGRESSION MODEL IS BEING MADE 

In [63]:
import joblib

# 1. Package all the lookup dictionaries and model pieces together
model_assets = {
    'preprocessor': preprocessor,
    'model': best_xgb,  # Your best tuned XGBoost model
    'team_to_id': team_to_id,
    'team_avg_goals_dict': team_avg_goals_dict,
    'global_avg_goals': global_avg_goals,
    'team_wc_exp_dict': team_wc_exp_dict,
    'team_pen_exp_dict': team_pen_exp_dict,
    'team_disc_score_dict': team_disc_score_dict,
    'global_disc_score': global_disc_score,
    'median_xg': median_xg,
    'median_attendance': median_attendance
}

# 2. Save the master asset file to disk
joblib.dump(model_assets, 'world_cup_model_assets.pkl')

# 3. Extract and save the unique 2026 World Cup teams list for your UI dropdowns
teams_2026 = sorted(list(set(df_shedule['home_team'].dropna().unique()).union(set(df_shedule['away_team'].dropna().unique()))))
joblib.dump(teams_2026, 'teams_2026.pkl')

print("✅ Notebook assets successfully exported! You can now build the Streamlit app.")

✅ Notebook assets successfully exported! You can now build the Streamlit app.


In [69]:
import joblib

# Gather all unique host names from your historical matches AND 2026 schedule datasets
historical_hosts = df_matches['Host'].dropna().unique().tolist()
future_hosts = df_shedule['Host'].dropna().unique().tolist() if 'Host' in df_shedule.columns else ['USA', 'Mexico', 'Canada']

# Combine them uniquely and sort alphabetically
all_hosts_list = sorted(list(set(historical_hosts + future_hosts)))

model_assets = {
    'preprocessor': preprocessor,
    'model': best_xgb,  
    'team_to_id': team_to_id,
    'team_avg_goals_dict': team_avg_goals_dict,
    'global_avg_goals': global_avg_goals,
    'team_wc_exp_dict': team_wc_exp_dict,
    'team_pen_exp_dict': team_pen_exp_dict,
    'team_disc_score_dict': team_disc_score_dict,
    'global_disc_score': global_disc_score,
    'median_xg': median_xg,
    'median_attendance': median_attendance,
    'all_hosts_list': all_hosts_list # <-- Adding the exact host names from your dataset here!
}

# Save it
joblib.dump(model_assets, 'world_cup_model_assets.pkl')

# Save 2026 teams list
teams_2026 = sorted(list(set(df_shedule['home_team'].dropna().unique()).union(set(df_shedule['away_team'].dropna().unique()))))
joblib.dump(teams_2026, 'teams_2026.pkl')

print("✅ Updated assets saved with your dataset's exact host list!")

✅ Updated assets saved with your dataset's exact host list!
